In [1]:
# inlegalbert_kg_rag_rrc.py  (KNOWLEDGE GRAPH RAG REVISION)
#
# Architecture:
#   InLegalBERT  →  BiLSTM  →  Multi-Head Attention Pooling  →  Linear  →  CRF
#   +
#   Knowledge Graph (KG) with RST-based edges
#   +
#   Uncertainty-triggered KG Retrieval + Graph Attention Fusion
#
# NEW vs BASELINE (inlegalbert_bilstm_mha_crf_rrc_v2.py):
#   1. KnowledgeGraph class: stores sentence embeddings as nodes per label,
#      builds RST-style edges (cosine-similarity proxy) within and across roles.
#   2. UncertaintyEstimator: computes entropy H(p_i) from softmax probabilities.
#   3. KGRetriever: similarity search across all label subgraphs, top-K selection,
#      1-hop graph expansion, subgraph G_i construction.
#   4. GraphAttentionFusion: weighted aggregation of neighbour embeddings,
#      alpha_j ∝ similarity × edge_weight; fused vector v_i.
#   5. KGAugmentedModel wraps the base model, runs first-pass CRF, checks
#      uncertainty, conditionally retrieves KG context, fuses h* = h_i + v_i,
#      re-runs CRF emission head.
#   6. Two-phase training:
#        Phase A – train base model (same as v2) + build KG from train embeddings.
#        Phase B – fine-tune KGAugmentedModel end-to-end (KG frozen, fusion trained).
#   7. All anti-overfitting settings from v2 retained.

import os, json, random, time, math
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_kg_rag_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60          # Phase A: base model training
NUM_EPOCHS_KG   = 20          # Phase B: KG-augmented fine-tuning
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

# BERT freeze / layer-wise LR decay
BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

# Sentence-level BiLSTM
SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2

# Multi-Head Attention Pooling
MHA_HEADS    = 4
MHA_DROPOUT  = 0.1

# Context-enrichment BiLSTM
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

# Auxiliary loss
AUX_CE_WEIGHT    = 0.2
LABEL_SMOOTHING  = 0.1

# Early stopping
ES_PATIENCE  = 10
ES_MIN_DELTA = 1e-4

WARMUP_RATIO = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD = 0.05

# ── KG-RAG specific ────────────────────────────────────────
KG_TOP_K            = 3      # top-K subgraphs to retrieve
KG_TOP_NODES        = 5      # nodes per subgraph to retrieve
KG_HOP              = 1      # graph expansion hops
UNCERTAINTY_THRESH  = 0.7    # entropy threshold (0-1 normalised) above which KG is used
RARE_ALWAYS_KG      = True   # always use KG for rare-class candidates
KG_FUSION_DIM       = 256    # sent_out_dim = SENT_LSTM_HIDDEN * 2

# RST edge similarity thresholds (proxy; real RST parser optional)
RST_INTRA_THRESH    = 0.6    # within same label subgraph
RST_CROSS_THRESH    = 0.5    # across different labels

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)

        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)

        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# BASE MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    ):
        super().__init__()

        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size    # 768
        self.dropout  = nn.Dropout(dropout)

        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        self.sent_out_dim = sent_lstm_hidden * 2        # 256

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim = self.sent_out_dim,
            num_heads  = mha_heads,
            dropout    = mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size    = self.sent_out_dim,
            hidden_size   = ctx_lstm_hidden,
            num_layers    = ctx_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2          # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )

        self.crf = CRF(num_tags=num_labels, batch_first=True)

        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100,
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total   = len(encoder_layers)
        n_trained = n_total - n_freeze
        print(f"\n❄️  BERT layers frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT layers trainable: layers {n_freeze}-{n_total-1} + pooler.\n")

    def encode_sentences(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        lengths:        torch.Tensor = None,
    ) -> torch.Tensor:
        """Returns sentence-level embeddings: (B, T, sent_out_dim)."""
        B, T, L = input_ids.shape
        N = B * T

        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)

        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)

        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)

        token_embs_all = self.dropout(token_embs_all)

        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)

        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False

        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)

        return sent_vecs.view(B, T, -1)

    def get_emissions(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        lengths:        torch.Tensor = None,
    ):
        """Returns (sent_vecs, ctx_out, emissions) for external use."""
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        sent_vecs_drop = self.dropout(sent_vecs)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs_drop, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs_drop)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
    ):
        _, _, emissions = self.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")

            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(
                emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),
            )

            loss = crf_loss + AUX_CE_WEIGHT * ce_loss
            return loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# KNOWLEDGE GRAPH
# ═══════════════════════════════════════════════════════════
class KnowledgeGraph:
    """
    Stores sentence embeddings as nodes organised per-label (subgraph).
    Edges are built using cosine-similarity as an RST proxy.

    Node structure:
        nodes[label_id] = list of {"emb": Tensor(D,), "text": str}

    Edge structure (per subgraph):
        intra_edges[label_id] = list of (i, j, weight)   # within same label
        cross_edges = list of (label_i, node_i, label_j, node_j, weight)
    """
    def __init__(self, emb_dim=KG_FUSION_DIM):
        self.emb_dim     = emb_dim
        self.nodes       = defaultdict(list)     # label_id -> list of node dicts
        self.intra_edges = defaultdict(list)     # label_id -> list of (i,j,w)
        self.cross_edges = []                    # (li,ni, lj,nj, w)
        self._stacked    = {}                    # label_id -> Tensor (N, D) on CPU

    # ── Population ──────────────────────────────────────────
    def add_nodes(self, embeddings: torch.Tensor, label_ids: list, texts: list = None):
        """Add sentence embeddings collected during training."""
        embs = embeddings.detach().cpu()
        for k, (emb, lid) in enumerate(zip(embs, label_ids)):
            text = texts[k] if texts is not None else ""
            self.nodes[lid].append({"emb": emb, "text": text})
        self._stacked = {}  # invalidate cache

    def build_edges(self,
                    intra_thresh=RST_INTRA_THRESH,
                    cross_thresh=RST_CROSS_THRESH,
                    max_intra_edges_per_node=5,
                    max_cross_edges=2000):
        """
        Build edges:
          INTRA: consecutive + high-sim pairs within same-label subgraph (RST proxy).
          CROSS: top-sim pairs across labels (discourse bridges).
        """
        print("  Building KG edges ...")
        self._stacked = {}
        self.intra_edges = defaultdict(list)
        self.cross_edges = []

        # ── Intra-label edges ─────────────────────────────
        for lid, node_list in self.nodes.items():
            N = len(node_list)
            if N < 2:
                continue
            embs = torch.stack([n["emb"] for n in node_list])   # (N, D)
            embs_norm = F.normalize(embs, dim=-1)
            sim_mat = torch.mm(embs_norm, embs_norm.T)          # (N, N)

            # consecutive edges always added
            for i in range(N - 1):
                w = float(sim_mat[i, i + 1].item())
                self.intra_edges[lid].append((i, i + 1, max(0.0, w)))

            # high-sim non-consecutive edges
            for i in range(N):
                sims = sim_mat[i].clone()
                sims[max(0, i-1):i+2] = -1   # mask out consecutive
                count = 0
                while count < max_intra_edges_per_node:
                    j = int(sims.argmax().item())
                    if sims[j] < intra_thresh:
                        break
                    w = float(sims[j].item())
                    self.intra_edges[lid].append((i, j, w))
                    sims[j] = -1
                    count += 1

            self._stacked[lid] = embs   # cache

        # ── Cross-label edges (discourse bridges) ─────────
        label_ids = list(self.nodes.keys())
        cross_count = 0
        for a in range(len(label_ids)):
            if cross_count >= max_cross_edges:
                break
            for b in range(a + 1, len(label_ids)):
                if cross_count >= max_cross_edges:
                    break
                la, lb = label_ids[a], label_ids[b]
                embs_a = self._get_stacked(la)
                embs_b = self._get_stacked(lb)
                if embs_a is None or embs_b is None:
                    continue
                na_norm = F.normalize(embs_a, dim=-1)
                nb_norm = F.normalize(embs_b, dim=-1)
                sim_mat = torch.mm(na_norm, nb_norm.T)          # (Na, Nb)
                high = (sim_mat >= cross_thresh).nonzero(as_tuple=False)
                for pair in high[:50]:
                    ni, nj = int(pair[0]), int(pair[1])
                    w = float(sim_mat[ni, nj].item())
                    self.cross_edges.append((la, ni, lb, nj, w))
                    cross_count += 1

        n_intra = sum(len(v) for v in self.intra_edges.values())
        print(f"  KG: {sum(len(v) for v in self.nodes.values())} nodes | "
              f"{n_intra} intra-edges | {len(self.cross_edges)} cross-edges")

    def _get_stacked(self, lid):
        if lid not in self._stacked:
            if lid not in self.nodes or not self.nodes[lid]:
                return None
            self._stacked[lid] = torch.stack([n["emb"] for n in self.nodes[lid]])
        return self._stacked[lid]

    def save(self, path):
        """Serialise KG to disk."""
        data = {
            "nodes": {str(k): [{"emb": n["emb"].tolist(), "text": n["text"]}
                                for n in v]
                      for k, v in self.nodes.items()},
            "intra_edges": {str(k): v for k, v in self.intra_edges.items()},
            "cross_edges": self.cross_edges,
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"  KG saved to {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM):
        kg = cls(emb_dim=emb_dim)
        with open(path) as f:
            data = json.load(f)
        for k, node_list in data["nodes"].items():
            lid = int(k)
            for n in node_list:
                kg.nodes[lid].append({"emb": torch.tensor(n["emb"]), "text": n["text"]})
        for k, edges in data["intra_edges"].items():
            kg.intra_edges[int(k)] = [tuple(e) for e in edges]
        kg.cross_edges = [tuple(e) for e in data["cross_edges"]]
        print(f"  KG loaded from {path}")
        return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    """Normalised entropy: H(p) / log(C) ∈ [0,1]."""
    def __init__(self, num_classes=NUM_LABELS, threshold=UNCERTAINTY_THRESH):
        self.log_C     = math.log(num_classes)
        self.threshold = threshold

    def entropy(self, logits: torch.Tensor) -> torch.Tensor:
        """logits: (..., C) → entropy (...,) in [0,1]."""
        probs = F.softmax(logits, dim=-1)
        eps   = 1e-9
        H     = -(probs * (probs + eps).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits: torch.Tensor) -> torch.Tensor:
        """Returns boolean mask of same shape as logits[..., 0]."""
        return self.entropy(logits) > self.threshold

    def top_label(self, logits: torch.Tensor) -> torch.Tensor:
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# KG RETRIEVER
# ═══════════════════════════════════════════════════════════
class KGRetriever:
    """
    Given a query embedding h_i:
      1. Similarity search: score each label subgraph.
      2. Select top-K subgraphs.
      3. Within each subgraph, find top-N nodes.
      4. 1-hop graph expansion via intra- and cross-edges.
      5. Return list of (embedding, weight) pairs = subgraph G_i.
    """
    def __init__(self, kg: KnowledgeGraph,
                 top_k=KG_TOP_K,
                 top_nodes=KG_TOP_NODES,
                 hop=KG_HOP):
        self.kg        = kg
        self.top_k     = top_k
        self.top_nodes = top_nodes
        self.hop       = hop

    def retrieve(self, h_i: torch.Tensor, rare_ids: list = None,
                 first_pass_label: int = None) -> list:
        """
        h_i: (D,) query embedding (CPU).
        Returns list of (emb: Tensor(D,), weight: float).
        """
        h_norm = F.normalize(h_i.unsqueeze(0), dim=-1)   # (1, D)

        # Step 1: score each subgraph
        subgraph_scores = {}
        for lid in self.kg.nodes:
            embs = self.kg._get_stacked(lid)
            if embs is None or embs.shape[0] == 0:
                continue
            embs_norm = F.normalize(embs, dim=-1)         # (N, D)
            sims = torch.mv(embs_norm, h_norm.squeeze(0)) # (N,)
            subgraph_scores[lid] = float(sims.max().item())

        # Step 2: top-K subgraphs
        sorted_sgs = sorted(subgraph_scores.items(), key=lambda x: -x[1])
        selected   = [lid for lid, _ in sorted_sgs[:self.top_k]]

        results = []   # list of (emb, weight)

        for lid in selected:
            embs = self.kg._get_stacked(lid)
            if embs is None:
                continue
            embs_norm = F.normalize(embs, dim=-1)
            sims = torch.mv(embs_norm, h_norm.squeeze(0))

            # Step 3: top-N nodes in this subgraph
            k = min(self.top_nodes, embs.shape[0])
            top_idx = sims.topk(k).indices.tolist()
            top_sims = sims.topk(k).values.tolist()

            seed_set = set(top_idx)

            # Step 4: 1-hop expansion via intra-edges
            if self.hop >= 1:
                for idx in list(seed_set):
                    for (i, j, w) in self.kg.intra_edges.get(lid, []):
                        if i == idx and j not in seed_set:
                            seed_set.add(j)
                        elif j == idx and i not in seed_set:
                            seed_set.add(i)

                # cross-edges
                for (la, ni, lb, nj, w) in self.kg.cross_edges:
                    if la == lid and ni in seed_set:
                        cross_embs = self.kg._get_stacked(lb)
                        if cross_embs is not None and nj < cross_embs.shape[0]:
                            results.append((cross_embs[nj], w))
                    elif lb == lid and nj in seed_set:
                        cross_embs = self.kg._get_stacked(la)
                        if cross_embs is not None and ni < cross_embs.shape[0]:
                            results.append((cross_embs[ni], w))

            # Collect seed nodes
            for idx, sim in zip(top_idx, top_sims):
                results.append((embs[idx], float(sim)))

        return results  # list of (Tensor(D), float)


# ═══════════════════════════════════════════════════════════
# GRAPH ATTENTION FUSION
# ═══════════════════════════════════════════════════════════
class GraphAttentionFusion(nn.Module):
    """
    Given h_i (query) and a set of (emb_j, weight_j) neighbours,
    compute attention weights α_j ∝ similarity(h_i, emb_j) × edge_weight_j
    and aggregate: v_i = Σ α_j * emb_j.

    The fused vector has the same dim as h_i (sent_out_dim = 256).
    """
    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT):
        super().__init__()
        self.emb_dim = emb_dim
        self.proj_q  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k  = nn.Linear(emb_dim, emb_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5

    def forward(self, h_i: torch.Tensor,
                neighbours: list,
                device=None) -> torch.Tensor:
        """
        h_i        : (D,)
        neighbours : list of (Tensor(D,), float weight)
        Returns    : v_i (D,)
        """
        if not neighbours:
            return torch.zeros_like(h_i)

        if device is None:
            device = h_i.device

        embs    = torch.stack([nb[0] for nb in neighbours]).to(device)  # (K, D)
        weights = torch.tensor([nb[1] for nb in neighbours],
                               device=device, dtype=torch.float)         # (K,)

        q = self.proj_q(h_i.unsqueeze(0))                   # (1, D)
        k = self.proj_k(embs)                                # (K, D)

        dot = torch.mv(k, q.squeeze(0)) * self.scale        # (K,)
        alpha = F.softmax(dot * weights, dim=0)             # (K,)  ∝ sim×weight
        alpha = self.dropout(alpha)

        v_i = (alpha.unsqueeze(-1) * embs).sum(dim=0)       # (D,)
        return v_i


# ═══════════════════════════════════════════════════════════
# KG-AUGMENTED MODEL
# ═══════════════════════════════════════════════════════════
class KGAugmentedModel(nn.Module):
    """
    Wraps the base InLegalBERT_BiLSTM_MHA_CRF model and adds:
      1. First-pass emission head → uncertainty check.
      2. Conditional KG retrieval.
      3. Graph attention fusion: h* = h_i + v_i.
      4. Project fused h* back to ctx_out_dim then re-run classifier → CRF.

    During training (Phase B), the base model weights continue to be
    updated; the KG is frozen (no grad); only GraphAttentionFusion +
    fusion_proj are newly trained.
    """
    def __init__(
        self,
        base_model: InLegalBERT_BiLSTM_MHA_CRF,
        kg:         KnowledgeGraph,
        rare_ids:   list,
        retriever:  KGRetriever = None,
    ):
        super().__init__()
        self.base      = base_model
        self.kg        = kg
        self.rare_ids  = rare_ids
        self.retriever = retriever or KGRetriever(kg)
        self.uncertainty = UncertaintyEstimator()

        sent_dim = base_model.sent_out_dim          # 256
        ctx_dim  = base_model.ctx_out_dim           # 128

        self.gat_fusion  = GraphAttentionFusion(emb_dim=sent_dim)
        # Project fused sentence vector (sent_dim) → ctx_dim for classifier reuse
        self.fusion_proj = nn.Sequential(
            nn.Linear(sent_dim * 2, ctx_dim),       # cat(h_i, v_i) → ctx_dim
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )
        # Extra CRF emission head for fused representations
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS),
        )
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss    = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100
        )

    def _kg_fuse_batch(
        self,
        sent_vecs:  torch.Tensor,   # (B, T, sent_dim)
        emissions:  torch.Tensor,   # (B, T, C)
        lengths:    torch.Tensor,   # (B,)
        device,
    ) -> torch.Tensor:
        """
        For each sentence in the batch, check uncertainty and
        optionally retrieve+fuse KG context.
        Returns fused_sent_vecs (B, T, sent_dim).
        """
        B, T, sent_dim = sent_vecs.shape
        fused = sent_vecs.clone()

        uncertain_mask = self.uncertainty.is_uncertain(emissions)  # (B, T)
        top_labels     = self.uncertainty.top_label(emissions)     # (B, T)

        for b in range(B):
            n = int(lengths[b].item())
            for t in range(n):
                uncertain = bool(uncertain_mask[b, t].item())
                pred_lbl  = int(top_labels[b, t].item())
                is_rare   = pred_lbl in self.rare_ids

                if not (uncertain or (RARE_ALWAYS_KG and is_rare)):
                    continue  # confident majority-class: use h_i directly

                h_i = sent_vecs[b, t].detach().cpu()  # KG is on CPU
                neighbours = self.retriever.retrieve(
                    h_i,
                    rare_ids=self.rare_ids,
                    first_pass_label=pred_lbl,
                )

                if not neighbours:
                    continue

                v_i = self.gat_fusion(
                    sent_vecs[b, t],      # keep on GPU for grads
                    neighbours,
                    device=device,
                )                         # (sent_dim,)

                # h* = concat[h_i, v_i] → project
                # Store back as cat for fusion_proj later
                fused[b, t] = sent_vecs[b, t] + v_i   # residual fusion

        return fused  # (B, T, sent_dim)

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
    ):
        device = input_ids.device

        # ── Step 1: first-pass base model ─────────────────
        sent_vecs, ctx_out, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )

        # ── Step 2: uncertainty + KG retrieval + fusion ───
        fused_sent = self._kg_fuse_batch(
            sent_vecs, base_emissions, lengths, device
        )  # (B, T, sent_dim)

        # ── Step 3: run context BiLSTM on fused sent vecs ─
        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _  = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)

        fused_ctx = self.base.dropout(fused_ctx)

        # ── Step 4: fusion classifier + CRF ───────────────
        fused_emissions = self.fusion_classifier(fused_ctx)
        fused_emissions = torch.nan_to_num(fused_emissions, nan=0.0,
                                            posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = fused_emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emissions.shape[:2], dtype=torch.bool,
                              device=device)

        # ── Combine base + fused emissions (ensemble) ─────
        combined_emissions = (base_emissions + fused_emissions) / 2.0

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            # Base CRF loss
            base_crf_loss = -self.base.crf(
                base_emissions, safe_labels, mask=mask, reduction="mean"
            )
            # Fused CRF loss
            fused_crf_loss = -self.fusion_crf(
                fused_emissions, safe_labels, mask=mask, reduction="mean"
            )
            # Auxiliary CE on combined
            B2, T2, C = combined_emissions.shape
            ce_loss = self.ce_loss(
                combined_emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),
            )

            loss = (base_crf_loss + fused_crf_loss) / 2.0 + AUX_CE_WEIGHT * ce_loss
            return loss, combined_emissions
        else:
            decoded = self.fusion_crf.decode(fused_emissions, mask=mask)
            return decoded, fused_emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]
    cls_report = classification_report(
        str_trues, str_preds, labels=LABELS, digits=4, zero_division=0,
    )
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": weighted_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec,
        "weighted_recall": weighted_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\n  Trainable: {total_trainable:,} | Frozen: {total_frozen:,}")
    return total_trainable, total_frozen


# ═══════════════════════════════════════════════════════════
# TRAINER  (Phase A: base model)
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        param_groups = []

        param_groups.append({
            "params": list(self.model.bert.pooler.parameters()),
            "lr": BERT_LR, "weight_decay": WEIGHT_DECAY,
        })

        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters() if p.requires_grad]
            if params:
                param_groups.append({
                    "params": params, "lr": lr_i, "weight_decay": WEIGHT_DECAY,
                })

        head_modules = [
            self.model.sent_bilstm, self.model.mha_pooling,
            self.model.sent_layer_norm, self.model.ctx_bilstm,
            self.model.classifier, self.model.crf,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))
        param_groups.append({
            "params": head_params, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY,
        })

        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attn, ttype, labels, lengths in loader:
                input_ids = input_ids.to(self.device)
                attn      = attn.to(self.device)
                ttype     = ttype.to(self.device)
                labels    = labels.to(self.device)
                lengths   = lengths.to(self.device)
                loss, _   = self.model(input_ids, attn, ttype,
                                       labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total_loss += loss.item(); n += 1
        return total_loss / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attn, ttype, labels, lengths in loader:
                input_ids = input_ids.to(self.device)
                attn      = attn.to(self.device)
                ttype     = ttype.to(self.device)
                lengths   = lengths.to(self.device)
                decoded, _ = self.model(input_ids, attn, ttype,
                                        labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents     = len(all_trues)
            infer_info  = {
                "total_inference_time_s":     total_infer,
                "latency_per_document_ms":    total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS_BASE):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                   shuffle=True, collate_fn=collate_rrc)
        optimizer   = self.build_optimizer()
        total_steps = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler   = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )

        early_stopper = EarlyStopping()
        history = []
        best_f1, best_state = -1.0, None

        total_start = time.time()
        actual_epochs = 0

        for epoch in range(1, num_epochs + 1):
            actual_epochs = epoch
            self.model.train()
            running_loss, n_steps, nan_steps = 0.0, 0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype,
                                     labels=labels, lengths=lengths)

                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1; optimizer.zero_grad(); continue

                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()

                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
                n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            val_loss       = self.compute_val_loss(dev_dataset)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[Base] Epoch {epoch:03d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | val_loss: {val_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | ES: {early_stopper.counter}/{early_stopper.patience}"
            )

            history.append({
                "epoch": epoch, "phase": "base",
                "train_loss": avg_train_loss, "val_loss": val_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_micro_f1": val_metrics["micro_f1"],
                "val_weighted_f1": val_metrics["weighted_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "base_history.csv"), index=False)

        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")

        if tokenizer:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state or self.model.state_dict(),
                       os.path.join(BEST_MODEL_DIR, "base_model.bin"))
            print(f"  Base model saved to {BEST_MODEL_DIR}/base_model.bin")

        return hist_df, total_time


# ═══════════════════════════════════════════════════════════
# KG BUILDER  (collect embeddings after Phase A)
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_knowledge_graph(base_model, train_docs, tokenizer,
                           device=DEVICE) -> KnowledgeGraph:
    """
    Run base_model encoder over training data and populate KG nodes.
    """
    print("\n🔨 Building Knowledge Graph from training embeddings ...")
    base_model.eval()
    base_model.to(device)

    kg = KnowledgeGraph(emb_dim=base_model.sent_out_dim)

    dummy_dataset = RRCDataset(train_docs, tokenizer)
    loader = DataLoader(dummy_dataset, batch_size=1, shuffle=False,
                        collate_fn=collate_rrc)

    for doc_idx, (ids, attn, ttype, labels, lengths) in enumerate(loader):
        ids     = ids.to(device)
        attn    = attn.to(device)
        ttype   = ttype.to(device)
        lengths = lengths.to(device)

        sent_vecs = base_model.encode_sentences(ids, attn, ttype)  # (1, T, D)
        sent_vecs = sent_vecs.squeeze(0)                            # (T, D)

        n = int(lengths[0].item())
        embs     = sent_vecs[:n].cpu()                              # (n, D)
        lab_ids  = labels[0, :n].tolist()

        kg.add_nodes(embs, lab_ids)

        if (doc_idx + 1) % 50 == 0:
            print(f"  Processed {doc_idx+1} / {len(loader)} docs")

    kg.build_edges()
    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    kg.save(kg_path)
    return kg


# ═══════════════════════════════════════════════════════════
# TRAINER  (Phase B: KG-augmented fine-tuning)
# ═══════════════════════════════════════════════════════════
class KGTrainer:
    def __init__(self, kg_model: KGAugmentedModel, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        # Fine-tune: base model (unfrozen layers) + new fusion modules
        new_params  = list(self.model.gat_fusion.parameters()) + \
                      list(self.model.fusion_proj.parameters()) + \
                      list(self.model.fusion_classifier.parameters()) + \
                      list(self.model.fusion_crf.parameters())

        base_trainable = [p for p in self.model.base.parameters() if p.requires_grad]

        param_groups = [
            {"params": new_params,      "lr": HEAD_LR,  "weight_decay": WEIGHT_DECAY},
            {"params": base_trainable,  "lr": BERT_LR,  "weight_decay": WEIGHT_DECAY},
        ]
        return torch.optim.AdamW(param_groups)

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                lengths = lengths.to(self.device)

                decoded, _ = self.model(ids, attn, ttype,
                                        labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].tolist())

        if measure_inference_time:
            total_infer = time.time() - infer_start
            n_sents = len(all_trues)
            infer_info = {
                "total_inference_time_s": total_infer,
                "latency_per_document_ms": total_infer / max(1, n_samples) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, total_infer),
            }
            with open(os.path.join(OUT_DIR, f"kg_inference_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def train(self, train_dataset, dev_dataset, rare_ids,
              num_epochs=NUM_EPOCHS_KG):
        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                   shuffle=True, collate_fn=collate_rrc)
        optimizer   = self.build_optimizer()
        total_steps = len(train_loader) * num_epochs
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler   = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )

        early_stopper = EarlyStopping(patience=5)
        history = []
        best_f1, best_state = -1.0, None
        total_start = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps, nan_steps = 0.0, 0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (ids, attn, ttype, labels, lengths) in enumerate(train_loader):
                ids     = ids.to(self.device)
                attn    = attn.to(self.device)
                ttype   = ttype.to(self.device)
                labels  = labels.to(self.device)
                lengths = lengths.to(self.device)

                loss, _ = self.model(ids, attn, ttype,
                                     labels=labels, lengths=lengths)

                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1; optimizer.zero_grad(); continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

                running_loss += loss.item()
                n_steps += 1

            epoch_time     = time.time() - epoch_start
            avg_train_loss = running_loss / max(1, n_steps)
            val_metrics    = self.evaluate(dev_dataset, rare_ids)

            print(
                f"[KG]   Epoch {epoch:02d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"time: {epoch_time:.1f}s | ES: {early_stopper.counter}/{early_stopper.patience}"
            )

            history.append({
                "epoch": epoch, "phase": "kg",
                "train_loss": avg_train_loss,
                "val_macro_f1": val_metrics["macro_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_accuracy": val_metrics["accuracy"],
                "epoch_train_time_s": epoch_time,
            })

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best KG val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  KG early stopping at epoch {epoch}.\n"); break

        total_time = time.time() - total_start
        pd.DataFrame(history).to_csv(
            os.path.join(OUT_DIR, "kg_history.csv"), index=False
        )
        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "kg_model.bin"))
            print(f"\n✔ Best KG model saved (val_macro_f1={best_f1:.4f})")

        return pd.DataFrame(history), total_time


# ═══════════════════════════════════════════════════════════
# VISUALISATION HELPERS
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
        for tick in ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    labels = LABELS
    f1s    = [per_class_metrics[l]["f1"] for l in labels]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
              for l in labels]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(labels, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1 (KG-RAG)")
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def plot_combined_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)

    # Phase boundary
    boundary = len(base_df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.plot(all_df["global_epoch"], all_df["train_loss"],
            label="Train Loss", marker="o", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Combined Training Loss"); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(all_df["global_epoch"], all_df["val_macro_f1"],
            label="Val Macro-F1", marker="o", markersize=3)
    ax.plot(all_df["global_epoch"], all_df["val_rare_f1"],
            label="Val Rare-F1", marker="s", markersize=3)
    ax.axvline(boundary, color="red", linestyle="--", label="Phase B start")
    ax.set_title("Val F1 (Base → KG-RAG)"); ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUT_DIR, "combined_training_curves.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def print_metrics_table(dev_metrics, test_metrics,
                        base_time=None, kg_time=None,
                        total_trainable=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Macro-Recall",       "macro_recall"),
    ]
    print("\n" + "=" * 72)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + CRF + KG-RAG)")
    print("=" * 72)
    if total_trainable:
        print(f"  Trainable Parameters  : {total_trainable:,}")
    if base_time:
        print(f"  Phase A training time : {base_time/60:.1f} min")
    if kg_time:
        print(f"  Phase B training time : {kg_time/60:.1f} min")
    print("-" * 72)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 72)
    for label, key in rows:
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 72)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 64)
    print(f"  {'Label':<22} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 64)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 64)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT + BiLSTM + MHA + CRF  →  KG-RAG\n")
    print("Phase A: Train base model, build Knowledge Graph")
    print("Phase B: Fine-tune KG-Augmented model\n")

    # ── Data ──────────────────────────────────────────────
    print("Loading JSONL files ...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    freq_df = pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ])
    freq_df.to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer ...")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ════════════════════════════════════════════════════
    # PHASE A: Train base model
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model Training")
    print("=" * 60)

    base_model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    )

    base_trainer = BaseTrainer(base_model, device=DEVICE)
    base_hist_df, base_time = base_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        tokenizer=tokenizer, num_epochs=NUM_EPOCHS_BASE,
    )

    # ── Build KG from best base model ─────────────────
    print("\n" + "=" * 60)
    print("PHASE A→B: Building Knowledge Graph")
    print("=" * 60)

    kg_path = os.path.join(OUT_DIR, "knowledge_graph.json")
    if os.path.exists(kg_path):
        print("  Found existing KG, loading ...")
        kg = KnowledgeGraph.load(kg_path, emb_dim=base_model.sent_out_dim)
    else:
        kg = build_knowledge_graph(base_model, train_docs, tokenizer, DEVICE)

    # ════════════════════════════════════════════════════
    # PHASE B: KG-Augmented fine-tuning
    # ════════════════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: KG-Augmented Fine-Tuning")
    print("=" * 60)

    retriever = KGRetriever(kg, top_k=KG_TOP_K,
                            top_nodes=KG_TOP_NODES, hop=KG_HOP)

    kg_model = KGAugmentedModel(
        base_model = base_model,
        kg         = kg,
        rare_ids   = rare_ids,
        retriever  = retriever,
    )

    total_trainable, _ = count_parameters(kg_model)

    kg_trainer  = KGTrainer(kg_model, device=DEVICE)
    kg_hist_df, kg_time = kg_trainer.train(
        train_dataset, dev_dataset, rare_ids,
        num_epochs=NUM_EPOCHS_KG,
    )

    plot_combined_history(base_hist_df, kg_hist_df)

    # ════════════════════════════════════════════════════
    # EVALUATION
    # ════════════════════════════════════════════════════
    print("\nEvaluating on Dev set ...")
    dev_metrics = kg_trainer.evaluate(dev_dataset, rare_ids,
                                       split_name="dev",
                                       measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + KG-RAG\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])

    save_confusion_matrix(dev_metrics["cm"], "dev", rare_labels)
    save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set ...")
    test_metrics = kg_trainer.evaluate(test_dataset, rare_ids,
                                        split_name="test",
                                        measure_inference_time=True)
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + KG-RAG\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pred_df = pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    })
    pred_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = [
        "macro_f1", "micro_f1", "weighted_f1", "rare_f1",
        "macro_precision", "micro_precision", "weighted_precision", "rare_precision",
        "macro_recall", "micro_recall", "weighted_recall", "rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        "model": "InLegalBERT + BiLSTM + MHA + CRF + KG-RAG",
        "kg_config": {
            "top_k": KG_TOP_K, "top_nodes": KG_TOP_NODES,
            "hop": KG_HOP, "uncertainty_thresh": UNCERTAINTY_THRESH,
            "rare_always_kg": RARE_ALWAYS_KG,
            "intra_thresh": RST_INTRA_THRESH, "cross_thresh": RST_CROSS_THRESH,
        },
        "timing": {
            "phase_a_s": base_time, "phase_b_s": kg_time,
            "total_s": base_time + kg_time,
        },
        "rare_classes": rare_labels,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        base_time=base_time, kg_time=kg_time,
        total_trainable=total_trainable,
    )

    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT + BiLSTM + MHA + CRF  →  KG-RAG

Phase A: Train base model, build Knowledge Graph
Phase B: Fine-tune KG-Augmented model

Loading JSONL files ...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  ( 1055 samples) ← RARE
   NONE                  4.79%  ( 1377 samples) ← RARE

   Rare classes (10): ['RLC', 'ISSUE'

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + layers 0-7.
🔥 BERT layers trainable: layers 8-11 + pooler.

[Base] Epoch 001/60 | train_loss: 287.7585 | val_loss: 211.9501 | val_macro_f1: 0.0391 | val_rare_f1: 0.0000 | time: 65.0s | ES: 0/10
  ✔ New best val_macro_f1=0.0391
[Base] Epoch 002/60 | train_loss: 234.5406 | val_loss: 175.5814 | val_macro_f1: 0.0797 | val_rare_f1: 0.0000 | time: 65.3s | ES: 0/10
  ✔ New best val_macro_f1=0.0797
[Base] Epoch 003/60 | train_loss: 198.7177 | val_loss: 128.8831 | val_macro_f1: 0.2005 | val_rare_f1: 0.0560 | time: 65.5s | ES: 0/10
  ✔ New best val_macro_f1=0.2005
[Base] Epoch 004/60 | train_loss: 166.5206 | val_loss: 106.5727 | val_macro_f1: 0.2568 | val_rare_f1: 0.0970 | time: 65.0s | ES: 0/10
  ✔ New best val_macro_f1=0.2568
[Base] Epoch 005/60 | train_loss: 140.3391 | val_loss: 88.4431 | val_macro_f1: 0.2828 | val_rare_f1: 0.1195 | time: 64.8s | ES: 0/10
  ✔ New best val_macro_f1=0.2828
[Base] Epoch 006/60 | train_loss: 124.8879 | val_loss: 83.0902 | val

In [1]:
"""
kg_visualization_fixed.py  – FULLY FIXED + ENHANCED
=====================================================
Fixes:
  ✔ KeyError: 'label'  →  always set node attrs BEFORE add_edge
  ✔ Sparse subgraph    →  edge-aware sampling, guarded node creation
  ✔ t-SNE max_iter     →  already fixed, kept

New:
  ✔ Fig 0 – Publication-quality label-level KG (matches your reference image)
  ✔ Fig 1 – Sentence-level subgraph (sampled, coloured by role)
  ✔ Fig 2 – Cross-label adjacency heatmap
  ✔ Fig 3 – PCA + t-SNE embedding scatter
  ✔ Fig 4 – Node-count bar chart (rare classes highlighted)
"""

import json
import os
import math
import random
from collections import defaultdict

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ── CONFIG ────────────────────────────────────────────────
KG_JSON_PATH = "rrc_kg_rag_logs/knowledge_graph.json"
OUT_DIR      = "kg_figures"
os.makedirs(OUT_DIR, exist_ok=True)

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
id2label = {i: l for i, l in enumerate(LABELS)}
label2id = {l: i for i, l in enumerate(LABELS)}

# Color palette – one per rhetorical role
PALETTE = [
    "#4E79A7", "#F28E2B", "#E15759", "#76B7B2", "#59A14F",
    "#EDC948", "#B07AA1", "#FF9DA7", "#9C755F", "#BAB0AC",
    "#D37295", "#FABFD2", "#8CD17D",
]
LABEL_COLOR = {l: PALETTE[i % len(PALETTE)] for i, l in enumerate(LABELS)}

# Rare labels (dashed border in reference image)
RARE_LABELS = {"RLC", "ISSUE", "STA", "PRE_NOT_RELIED", "RPC"}


# ── LOAD KG ───────────────────────────────────────────────
def load_kg(path):
    with open(path) as f:
        kg = json.load(f)

    nodes_by_label = {
        int(k): [np.array(n["emb"]) for n in v]
        for k, v in kg["nodes"].items()
    }
    intra_edges = {
        int(k): [(int(a), int(b), float(w)) for a, b, w in v]
        for k, v in kg["intra_edges"].items()
    }
    cross_edges = [
        (int(a), int(b), int(c), int(d), float(w))
        for a, b, c, d, w in kg["cross_edges"]
    ]

    total_nodes = sum(len(v) for v in nodes_by_label.values())
    total_intra = sum(len(v) for v in intra_edges.values())
    print(f"Loaded KG: {total_nodes} nodes | {total_intra} intra-edges "
          f"| {len(cross_edges)} cross-edges")
    return nodes_by_label, intra_edges, cross_edges


# ════════════════════════════════════════════════════════
# FIG 0 — Label-level KG  (matches your reference image)
# ════════════════════════════════════════════════════════
def fig0_label_graph(nodes_by_label, cross_edges):
    """
    One node per rhetorical role.
    Node size  ∝ number of sentences with that role.
    Edge width ∝ number of cross-edges between roles.
    """
    print("[Fig 0] Label-level KG ...")
    G = nx.Graph()

    # ── Add nodes (all 13 roles) ───────────────────────
    for lid, lbl in id2label.items():
        count = len(nodes_by_label.get(lid, []))
        G.add_node(lbl, count=count, lid=lid)

    # ── Aggregate cross-edge weights ──────────────────
    edge_weight = defaultdict(float)
    for la, _, lb, _, w in cross_edges:
        a, b = id2label[la], id2label[lb]
        if a != b:
            key = tuple(sorted([a, b]))
            edge_weight[key] += w

    for (a, b), w in edge_weight.items():
        G.add_edge(a, b, weight=w)

    # ── Layout ────────────────────────────────────────
    pos = nx.circular_layout(G)

    # ── Visual sizes / widths ─────────────────────────
    counts  = [G.nodes[n]["count"] for n in G.nodes()]
    max_cnt = max(counts) if counts else 1
    node_sizes = [800 + 3000 * (c / max_cnt) for c in counts]

    all_w = [G[u][v]["weight"] for u, v in G.edges()]
    max_w = max(all_w) if all_w else 1
    edge_widths = [0.5 + 4.0 * (G[u][v]["weight"] / max_w) for u, v in G.edges()]
    edge_alphas = [0.3 + 0.5 * (G[u][v]["weight"] / max_w) for u, v in G.edges()]

    node_colors  = [LABEL_COLOR[n] for n in G.nodes()]
    node_edgecolors = [
        "#CC0000" if n in RARE_LABELS else "#333333"
        for n in G.nodes()
    ]
    linewidths = [2.5 if n in RARE_LABELS else 1.2 for n in G.nodes()]

    # ── Draw ──────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 10))
    ax.set_facecolor("#FAFAFA")
    fig.patch.set_facecolor("#FAFAFA")

    # Draw edges with individual alpha (workaround: draw one by one)
    for (u, v), ew, ea in zip(G.edges(), edge_widths, edge_alphas):
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        ax.plot([x0, x1], [y0, y1],
                color="#AAAAAA", linewidth=ew, alpha=ea, zorder=1)

    # Draw nodes
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_size=node_sizes,
        node_color=node_colors,
        edgecolors=node_edgecolors,
        linewidths=linewidths,
        alpha=0.88,
    )

    # Draw labels + counts below
    for node in G.nodes():
        x, y = pos[node]
        count = G.nodes[node]["count"]
        # Short label inside node
        short = node if len(node) <= 9 else node[:8] + "…"
        ax.text(x, y + 0.04, short,
                ha="center", va="center",
                fontsize=7.5, fontweight="bold", color="#111111", zorder=5)
        ax.text(x, y - 0.08, str(count),
                ha="center", va="center",
                fontsize=7, color="#444444", zorder=5)

    # Legend: rare vs common
    rare_patch   = mpatches.Patch(edgecolor="#CC0000", facecolor="none",
                                   linewidth=2, label="Rare class (dashed border)")
    common_patch = mpatches.Patch(edgecolor="#333333", facecolor="none",
                                   linewidth=1, label="Common class")
    ax.legend(handles=[rare_patch, common_patch],
              loc="lower right", fontsize=9, framealpha=0.7)

    ax.set_title(
        "Knowledge Graph – Label-Level View\n"
        "(node size ∝ sentence count · edge width ∝ cross-role similarity)",
        fontsize=13, fontweight="bold", pad=16
    )
    ax.axis("off")
    plt.tight_layout()
    path = f"{OUT_DIR}/fig0_label_graph.png"
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"  Saved {path}")


# ════════════════════════════════════════════════════════
# FIG 1 — Sentence-level subgraph (FIXED KeyError)
# ════════════════════════════════════════════════════════
def fig1_subgraph(nodes_by_label, intra_edges, cross_edges,
                  max_nodes_per_label=15, max_cross=80):
    """
    KEY FIX: always call G.add_node(nid, label=...) BEFORE G.add_edge().
    This ensures every node in the graph has the 'label' attribute,
    preventing the KeyError when iterating G.nodes().
    """
    print("[Fig 1] Sentence-level subgraph ...")
    G = nx.Graph()

    # ── Helper: safely add a node with guaranteed label attr ──
    def safe_add_node(nid, label_str):
        if nid not in G:
            G.add_node(nid, label=label_str)

    # ── Intra-label edges (sampled) ───────────────────
    for lid, node_list in nodes_by_label.items():
        lbl = id2label.get(lid, str(lid))
        edges = intra_edges.get(lid, [])
        if not edges:
            continue

        # Collect connected node indices
        connected = set()
        for i, j, _ in edges:
            connected.add(i)
            connected.add(j)

        # Sample at most max_nodes_per_label connected nodes
        sampled = set(random.sample(sorted(connected),
                                    min(max_nodes_per_label, len(connected))))

        # Add sampled nodes FIRST (with label attr)
        for idx in sampled:
            nid = f"{lbl}_{idx}"
            safe_add_node(nid, lbl)

        # Add only edges whose both endpoints are sampled
        for i, j, w in edges:
            if i in sampled and j in sampled:
                G.add_edge(f"{lbl}_{i}", f"{lbl}_{j}", weight=float(w))

    # ── Cross-label edges (sampled) ───────────────────
    cross_sample = random.sample(cross_edges, min(max_cross, len(cross_edges)))
    for la, ni, lb, nj, w in cross_sample:
        la_str, lb_str = id2label.get(la, str(la)), id2label.get(lb, str(lb))
        nid_a = f"{la_str}_{ni}"
        nid_b = f"{lb_str}_{nj}"
        # Only add cross-edge if BOTH nodes already exist in graph
        if nid_a in G and nid_b in G:
            G.add_edge(nid_a, nid_b, weight=float(w), cross=True)

    print(f"  Subgraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    if G.number_of_nodes() == 0:
        print("  Empty graph – skipping.")
        return

    # ── Colors (safe: use .get with default) ──────────
    colors = [
        LABEL_COLOR.get(G.nodes[n].get("label", "NONE"), "#CCCCCC")
        for n in G.nodes()
    ]

    pos = nx.spring_layout(G, k=2.5, seed=42)

    plt.figure(figsize=(12, 10))
    ax = plt.gca()
    ax.set_facecolor("#F8F8F8")

    # Draw cross-edges (grey, thin)
    cross_edge_list = [(u, v) for u, v in G.edges()
                       if G[u][v].get("cross", False)]
    nx.draw_networkx_edges(G, pos, edgelist=cross_edge_list,
                           edge_color="#BBBBBB", alpha=0.4, width=0.7, ax=ax)

    # Draw intra-edges (coloured)
    intra_edge_list = [(u, v) for u, v in G.edges()
                       if not G[u][v].get("cross", False)]
    nx.draw_networkx_edges(G, pos, edgelist=intra_edge_list,
                           edge_color="#888888", alpha=0.5, width=1.0, ax=ax)

    nx.draw_networkx_nodes(G, pos, node_size=60,
                           node_color=colors, alpha=0.85, ax=ax)

    # Legend
    patches = [mpatches.Patch(color=LABEL_COLOR[l], label=l) for l in LABELS
               if any(G.nodes[n].get("label") == l for n in G.nodes())]
    plt.legend(handles=patches, loc="upper right",
               fontsize=7, ncol=2, framealpha=0.75)
    plt.title("Sentence-Level Knowledge Subgraph (sampled)",
              fontsize=13, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    path = f"{OUT_DIR}/fig1_subgraph.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved {path}")


# ════════════════════════════════════════════════════════
# FIG 2 — Cross-label adjacency heatmap
# ════════════════════════════════════════════════════════
def fig2_adj(nodes_by_label, cross_edges):
    print("[Fig 2] Adjacency heatmap ...")
    n = len(LABELS)
    mat = np.zeros((n, n))
    for la, _, lb, _, w in cross_edges:
        if la < n and lb < n:
            mat[la, lb] += 1
            mat[lb, la] += 1

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(mat, xticklabels=LABELS, yticklabels=LABELS,
                cmap="YlOrRd", annot=False, linewidths=0.4,
                linecolor="#DDDDDD", ax=ax)
    ax.set_title("Cross-Label Edge Adjacency Matrix", fontsize=13, fontweight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    path = f"{OUT_DIR}/fig2_adjacency.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved {path}")


# ════════════════════════════════════════════════════════
# FIG 3 — PCA + t-SNE embedding scatter
# ════════════════════════════════════════════════════════
def fig3_embeddings(nodes_by_label, max_per_label=60):
    print("[Fig 3] Embedding scatter ...")
    X, y = [], []
    for lid, embs in nodes_by_label.items():
        sample = random.sample(embs, min(max_per_label, len(embs)))
        X.extend(sample)
        y.extend([lid] * len(sample))

    if not X:
        print("  No embeddings found – skipping.")
        return

    X = np.stack(X)
    colors = [LABEL_COLOR.get(id2label.get(yi, "NONE"), "#CCCCCC") for yi in y]

    # PCA
    Xp = PCA(n_components=2).fit_transform(X)

    # t-SNE
    perp = min(30, max(5, len(X) // 5))
    Xt = TSNE(n_components=2, perplexity=perp,
               max_iter=500, random_state=42).fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor("#FAFAFA")

    for ax, Xr, title in zip(axes, [Xp, Xt], ["PCA", "t-SNE"]):
        ax.set_facecolor("#F0F0F0")
        ax.scatter(Xr[:, 0], Xr[:, 1], c=colors, s=18, alpha=0.75, edgecolors="none")
        ax.set_title(f"Sentence Embeddings – {title}",
                     fontsize=12, fontweight="bold")
        ax.set_xticks([]); ax.set_yticks([])

    # Shared legend
    patches = [mpatches.Patch(color=LABEL_COLOR[l], label=l) for l in LABELS]
    fig.legend(handles=patches, loc="lower center", ncol=7,
               fontsize=7.5, framealpha=0.8, bbox_to_anchor=(0.5, -0.04))

    plt.suptitle("KG Node Embeddings by Rhetorical Role",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    path = f"{OUT_DIR}/fig3_embeddings.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved {path}")


# ════════════════════════════════════════════════════════
# FIG 4 — Node count bar chart
# ════════════════════════════════════════════════════════
def fig4_node_counts(nodes_by_label):
    print("[Fig 4] Node count bar ...")
    labels = [id2label[i] for i in range(len(LABELS)) if i in nodes_by_label]
    counts = [len(nodes_by_label[label2id[l]]) for l in labels]
    colors = [
        "#E15759" if l in RARE_LABELS else LABEL_COLOR[l]
        for l in labels
    ]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(labels, counts, color=colors, edgecolor="white", height=0.6)
    ax.bar_label(bars, padding=4, fontsize=8)
    ax.set_xlabel("Number of Sentence Nodes", fontsize=10)
    ax.set_title("KG Node Count per Rhetorical Role\n(red = rare class)",
                 fontsize=12, fontweight="bold")
    ax.grid(True, axis="x", alpha=0.3)
    ax.set_facecolor("#FAFAFA")
    plt.tight_layout()
    path = f"{OUT_DIR}/fig4_node_counts.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved {path}")


# ════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════
def main():
    random.seed(42)
    np.random.seed(42)

    nodes_by_label, intra_edges, cross_edges = load_kg(KG_JSON_PATH)

    fig0_label_graph(nodes_by_label, cross_edges)         # ⭐ publication-quality
    fig1_subgraph(nodes_by_label, intra_edges, cross_edges)  # fixed KeyError
    fig2_adj(nodes_by_label, cross_edges)
    fig3_embeddings(nodes_by_label)
    fig4_node_counts(nodes_by_label)

    print(f"\n✅  All figures saved in: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Loaded KG: 28739 nodes | 172341 intra-edges | 2000 cross-edges
[Fig 0] Label-level KG ...
  Saved kg_figures/fig0_label_graph.png
[Fig 1] Sentence-level subgraph ...
  Subgraph: 195 nodes, 17 edges
  Saved kg_figures/fig1_subgraph.png
[Fig 2] Adjacency heatmap ...
  Saved kg_figures/fig2_adjacency.png
[Fig 3] Embedding scatter ...
  Saved kg_figures/fig3_embeddings.png
[Fig 4] Node count bar ...
  Saved kg_figures/fig4_node_counts.png

✅  All figures saved in: kg_figures/


In [1]:
# inlegalbert_kg_rag_fixed.py
#
# ROOT CAUSE FIX:
#   The original Phase B was idle/slow because _kg_fuse_batch() looped over
#   every (b, t) sentence pair and called retriever.retrieve() one-at-a-time
#   on CPU — doing cosine similarity scans across the full KG for each sentence.
#   With large KG + 2-hop expansion, this is O(B*T * KG_nodes) per forward pass,
#   taking hours per epoch.
#
# SOLUTION:
#   1. Precompute a BATCHED GPU index: stack all KG node embeddings into a single
#      (N_total, D) tensor on GPU at the start of Phase B.
#   2. BatchedKGRetriever.retrieve_batch(): takes (B, T, D) sent_vecs GPU tensor,
#      computes cosine sim (B*T, N_total) in one matmul, selects top-K indices —
#      all on GPU, no Python loops over sentences.
#   3. The KG fusion then does a weighted sum using gathered embeddings.
#   4. 2-hop expansion replaced by a precomputed adjacency-weighted matrix
#      (also on GPU), applied once per batch.
#
# MINORITY F1 IMPROVEMENTS (on top of enhanced version):
#   - Manual minority weights further boosted (PRE_NOT_RELIED x10, ARG_RESP x6)
#   - KG contrastive margin tightened to 0.2 for harder pushing
#   - Prototype anchoring scale raised to learnable init 0.3
#   - Emission bias for rare classes: add +log_prior_correction to emissions
#   - CRF transition penalty: during decoding, penalise transitions FROM rare TO common
#     to force model to commit once it predicts rare
#   - Label-smoothing target replaced by KG-neighbour distribution for rare classes
#   - 60 epochs for Phase B, NO early stopping

import os, json, random, time, math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_kg_fixed_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS_BASE = 60
NUM_EPOCHS_KG   = 60          # Full 60 epochs, no early stopping
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2
MHA_HEADS    = 4
MHA_DROPOUT  = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

AUX_CE_WEIGHT    = 0.2
LABEL_SMOOTHING  = 0.1

# Phase A early stopping
ES_PATIENCE_BASE = 10
ES_MIN_DELTA     = 1e-4

WARMUP_RATIO = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD = 0.05

# ── KG specific ────────────────────────────────────────────
KG_TOP_K            = 8       # top-K nodes per query (batched)
UNCERTAINTY_THRESH  = 0.55    # more aggressive retrieval
RARE_ALWAYS_KG      = True
KG_FUSION_DIM       = 256     # = sent_out_dim

RST_INTRA_THRESH    = 0.55
RST_CROSS_THRESH    = 0.45

KG_CONTRASTIVE_WEIGHT = 0.20
KG_PROTO_WEIGHT_INIT  = 0.3
HOP_DECAY             = 0.5

# Max KG nodes to keep per label (memory cap)
KG_MAX_NODES_PER_LABEL = 500

CONFUSED_PAIRS = [
    ("ARG_PETITIONER", "ARG_RESPONDENT"),
    ("PRE_RELIED", "ANALYSIS"),
    ("RLC", "FAC"),
    ("RATIO", "ANALYSIS"),
]

ROLE_GROUPS = {
    "STRUCTURE":  ["PREAMBLE", "FAC", "ISSUE", "NONE"],
    "ARGUMENT":   ["ARG_PETITIONER", "ARG_RESPONDENT", "RLC"],
    "PRECEDENT":  ["PRE_RELIED", "PRE_NOT_RELIED", "STA"],
    "DECISION":   ["ANALYSIS", "RATIO", "RPC"],
}
LABEL_TO_GROUP = {}
for grp, lbls in ROLE_GROUPS.items():
    for lbl in lbls:
        LABEL_TO_GROUP[lbl] = grp

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"

# Boosted minority weights
MANUAL_MINORITY_WEIGHTS = {
    "PRE_NOT_RELIED": 10.0,
    "ARG_RESPONDENT":  6.0,
    "RLC":             4.0,
    "ARG_PETITIONER":  3.5,
    "ISSUE":           3.0,
    "PRE_RELIED":      2.5,
    "STA":             2.5,
    "RATIO":           2.0,
    "RPC":             1.8,
    "NONE":            1.5,
    "FAC":             1.0,
    "ANALYSIS":        0.8,
    "PREAMBLE":        0.7,
}


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING (Phase A only)
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE_BASE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


def compute_class_weights(docs, rare_ids):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    weights = []
    for i in range(NUM_LABELS):
        lbl    = id2label[i]
        count  = counts.get(i, 1)
        freq_w = total / (NUM_LABELS * count)
        manual_w = MANUAL_MINORITY_WEIGHTS.get(lbl, 1.0)
        w = max(freq_w, manual_w) if i in rare_ids else min(freq_w, manual_w)
        weights.append(w)
    w_tensor = torch.tensor(weights, dtype=torch.float)
    w_tensor = w_tensor / w_tensor.mean()
    print("\n📊 Class Weights:")
    for i, lbl in enumerate(LABELS):
        print(f"   {lbl:<20}  {w_tensor[i]:.3f}")
    return w_tensor


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents, padding="max_length", truncation=True,
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get("token_type_ids",
                                      torch.zeros_like(enc["input_ids"])),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim, num_heads=4, dropout=0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x, key_padding_mask=None):
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            attn_weights = attn_weights.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        attn_weights = self.attn_drop(F.softmax(attn_weights, dim=-1))
        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# BASE MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):
    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        class_weights    = None,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size=self.bert_dim, hidden_size=sent_lstm_hidden,
            num_layers=sent_lstm_layers, bidirectional=True,
            batch_first=True, dropout=dropout if sent_lstm_layers > 1 else 0.0,
        )
        self.sent_out_dim = sent_lstm_hidden * 2

        self.mha_pooling     = MultiHeadAttentionPooling(self.sent_out_dim, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(self.sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size=self.sent_out_dim, hidden_size=ctx_lstm_hidden,
            num_layers=ctx_lstm_layers, bidirectional=True,
            batch_first=True, dropout=dropout if ctx_lstm_layers > 1 else 0.0,
        )
        self.ctx_out_dim = ctx_lstm_hidden * 2

        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(self.ctx_out_dim, self.ctx_out_dim // 2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(self.ctx_out_dim // 2, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            weight=class_weights, label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

    def _freeze_bert_layers(self, n_freeze):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        for i in range(min(n_freeze, len(self.bert.encoder.layer))):
            for param in self.bert.encoder.layer[i].parameters():
                param.requires_grad = False
        n = len(self.bert.encoder.layer)
        print(f"\n❄️  BERT frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT trainable: layers {n_freeze}-{n-1} + pooler.\n")

    def encode_sentences(self, input_ids, attention_mask, token_type_ids, lengths=None):
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(input_ids=flat_ids[valid], attention_mask=flat_mask[valid],
                            token_type_ids=flat_types[valid])
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)
        token_embs_all = self.dropout(token_embs_all)
        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out = self.dropout(lstm_out)
        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def get_emissions(self, input_ids, attention_mask, token_type_ids, lengths=None):
        sent_vecs = self.encode_sentences(input_ids, attention_mask, token_type_ids)
        sv_drop   = self.dropout(sent_vecs)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sv_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            ctx_out, _ = self.ctx_bilstm(sv_drop)
        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)
        return sent_vecs, ctx_out, emissions

    def _make_mask(self, emissions, labels, lengths):
        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths): mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool, device=emissions.device)
        return mask

    def forward(self, input_ids, attention_mask, token_type_ids, labels=None, lengths=None):
        _, _, emissions = self.get_emissions(input_ids, attention_mask, token_type_ids, lengths)
        mask = self._make_mask(emissions, labels, lengths)
        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            crf_loss = -self.crf(emissions, safe, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(emissions.reshape(B2 * T2, C), labels.reshape(B2 * T2))
            return crf_loss + AUX_CE_WEIGHT * ce_loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# BATCHED GPU KG INDEX  ← THE KEY FIX
# ═══════════════════════════════════════════════════════════
class BatchedKGIndex:
    """
    Precomputes a flat GPU tensor of ALL KG node embeddings.
    retrieve_batch() computes similarity for the entire batch in ONE matmul —
    no Python loops over sentences, no CPU round-trips per sentence.

    Shape: kg_embs (N_total, D) on GPU
           kg_labels (N_total,) int — which label each node belongs to
    """
    def __init__(self, kg_nodes: dict, emb_dim: int, device: str,
                 max_per_label: int = KG_MAX_NODES_PER_LABEL):
        all_embs   = []
        all_labels = []
        self.label_offsets = {}   # lid → (start, end) in flat array
        idx = 0
        for lid in sorted(kg_nodes.keys()):
            node_list = kg_nodes[lid]
            # Sub-sample if too many (keep random subset for speed)
            if len(node_list) > max_per_label:
                node_list = random.sample(node_list, max_per_label)
            start = idx
            for node in node_list:
                all_embs.append(node["emb"])
                all_labels.append(lid)
                idx += 1
            self.label_offsets[lid] = (start, idx)

        if all_embs:
            emb_matrix = torch.stack(all_embs).to(device)          # (N, D)
            self.kg_embs_norm = F.normalize(emb_matrix, dim=-1)    # precomputed
            self.kg_embs_raw  = emb_matrix
            self.kg_labels    = torch.tensor(all_labels, dtype=torch.long, device=device)
        else:
            self.kg_embs_norm = torch.zeros(1, emb_dim, device=device)
            self.kg_embs_raw  = torch.zeros(1, emb_dim, device=device)
            self.kg_labels    = torch.zeros(1, dtype=torch.long, device=device)

        # Per-label prototype (mean embedding, normalised)
        self.prototypes = {}  # lid → (D,) on device
        for lid in sorted(kg_nodes.keys()):
            s, e = self.label_offsets.get(lid, (0, 0))
            if e > s:
                proto = self.kg_embs_raw[s:e].mean(dim=0)
                self.prototypes[lid] = F.normalize(proto, dim=-1)

        self.N      = self.kg_embs_norm.shape[0]
        self.D      = emb_dim
        self.device = device
        print(f"  BatchedKGIndex: {self.N} nodes, {len(self.label_offsets)} labels "
              f"(max {max_per_label}/label) on {device}")

    def retrieve_batch(self, sent_vecs: torch.Tensor, top_k: int = KG_TOP_K):
        """
        sent_vecs: (B, T, D)  already on GPU
        Returns:
          top_embs   (B, T, top_k, D)  — raw KG embeddings
          top_sims   (B, T, top_k)     — cosine similarities
          top_labels (B, T, top_k)     — label ids of retrieved nodes
        """
        B, T, D = sent_vecs.shape
        sv_flat  = sent_vecs.reshape(B * T, D)                     # (BT, D)
        sv_norm  = F.normalize(sv_flat, dim=-1)                    # (BT, D)

        # ONE matmul: (BT, D) x (D, N) → (BT, N)
        sim_matrix = torch.mm(sv_norm, self.kg_embs_norm.T)        # (BT, N)

        # top-K per query
        top_k_actual  = min(top_k, self.N)
        top_sims_flat, top_idx_flat = sim_matrix.topk(top_k_actual, dim=-1)
        # top_sims_flat: (BT, K), top_idx_flat: (BT, K)

        top_embs_flat   = self.kg_embs_raw[top_idx_flat]           # (BT, K, D)
        top_labels_flat = self.kg_labels[top_idx_flat]             # (BT, K)

        top_embs   = top_embs_flat.view(B, T, top_k_actual, D)
        top_sims   = top_sims_flat.view(B, T, top_k_actual)
        top_labels = top_labels_flat.view(B, T, top_k_actual)
        return top_embs, top_sims, top_labels

    def get_prototype_matrix(self) -> torch.Tensor:
        """Returns (C, D) prototype matrix on device, zeros for missing labels."""
        C = NUM_LABELS
        D = self.D
        proto = torch.zeros(C, D, device=self.device)
        for lid, p in self.prototypes.items():
            if lid < C:
                proto[lid] = p
        return proto


# ═══════════════════════════════════════════════════════════
# KG GRAPH (for building the index)
# ═══════════════════════════════════════════════════════════
class KGStore:
    """Lightweight storage: just nodes per label (no edge overhead at train time)."""
    def __init__(self, emb_dim=KG_FUSION_DIM):
        self.emb_dim = emb_dim
        self.nodes   = defaultdict(list)

    def add_nodes(self, embeddings, label_ids, texts=None):
        embs = embeddings.detach().cpu()
        for k, (emb, lid) in enumerate(zip(embs, label_ids)):
            text = texts[k] if texts is not None else ""
            self.nodes[lid].append({"emb": emb, "text": text})

    def save(self, path):
        data = {"nodes": {str(k): [{"emb": n["emb"].tolist(), "text": n["text"]}
                                    for n in v]
                          for k, v in self.nodes.items()}}
        with open(path, "w") as f: json.dump(data, f)
        print(f"  KGStore saved → {path}")

    @classmethod
    def load(cls, path, emb_dim=KG_FUSION_DIM):
        kg = cls(emb_dim=emb_dim)
        with open(path) as f: data = json.load(f)
        for k, node_list in data["nodes"].items():
            lid = int(k)
            for n in node_list:
                kg.nodes[lid].append({"emb": torch.tensor(n["emb"]), "text": n["text"]})
        total = sum(len(v) for v in kg.nodes.values())
        print(f"  KGStore loaded from {path}: {total} nodes")
        return kg


# ═══════════════════════════════════════════════════════════
# UNCERTAINTY ESTIMATOR
# ═══════════════════════════════════════════════════════════
class UncertaintyEstimator:
    def __init__(self, num_classes=NUM_LABELS, threshold=UNCERTAINTY_THRESH):
        self.log_C     = math.log(num_classes)
        self.threshold = threshold

    def entropy(self, logits):
        probs = F.softmax(logits, dim=-1)
        H = -(probs * (probs + 1e-9).log()).sum(dim=-1)
        return H / self.log_C

    def is_uncertain(self, logits):
        return self.entropy(logits) > self.threshold

    def top_label(self, logits):
        return logits.argmax(dim=-1)


# ═══════════════════════════════════════════════════════════
# BATCHED GRAPH ATTENTION FUSION
# ═══════════════════════════════════════════════════════════
class BatchedGraphAttentionFusion(nn.Module):
    """
    Fully batched: inputs are GPU tensors, no Python loops.

    h_vecs    : (B, T, D)
    top_embs  : (B, T, K, D)
    top_sims  : (B, T, K)
    fusion_mask: (B, T)  — True where fusion should occur
    Returns v  : (B, T, D)
    """
    def __init__(self, emb_dim=KG_FUSION_DIM, dropout=DROPOUT):
        super().__init__()
        self.proj_q = nn.Linear(emb_dim, emb_dim, bias=False)
        self.proj_k = nn.Linear(emb_dim, emb_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale   = emb_dim ** -0.5
        self.gate    = nn.Sequential(nn.Linear(emb_dim * 2, 1), nn.Sigmoid())

    def forward(self, h_vecs, top_embs, top_sims, fusion_mask):
        B, T, D = h_vecs.shape
        K       = top_embs.shape[2]

        # Query: (B, T, D) → (B, T, 1, D)
        q = self.proj_q(h_vecs).unsqueeze(2)        # (B, T, 1, D)
        # Key: (B, T, K, D)
        k = self.proj_k(top_embs)                   # (B, T, K, D)

        # dot: (B, T, K)
        dot   = (q * k).sum(dim=-1) * self.scale    # (B, T, K)
        alpha = F.softmax(dot * top_sims, dim=-1)   # (B, T, K)
        alpha = self.dropout(alpha)

        # weighted sum: (B, T, D)
        v = (alpha.unsqueeze(-1) * top_embs).sum(dim=2)   # (B, T, D)

        # gate
        gate_val = self.gate(torch.cat([h_vecs, v], dim=-1)).squeeze(-1)  # (B, T)
        v = gate_val.unsqueeze(-1) * v                                     # (B, T, D)

        # apply only where fusion_mask is True
        out = h_vecs.clone()
        out[fusion_mask] = h_vecs[fusion_mask] + v[fusion_mask]
        return out


# ═══════════════════════════════════════════════════════════
# KG CONTRASTIVE LOSS (batched)
# ═══════════════════════════════════════════════════════════
class KGContrastiveLoss(nn.Module):
    def __init__(self, margin=0.2, weight=KG_CONTRASTIVE_WEIGHT):
        super().__init__()
        self.margin = margin
        self.weight = weight

    def forward(self, sent_vecs, pred_labels, true_labels, proto_matrix, device):
        """
        sent_vecs   : (B, T, D)
        pred_labels : (B, T)
        true_labels : (B, T)  (-100 = ignore)
        proto_matrix: (C, D)
        """
        valid   = true_labels != -100                              # (B, T)
        mismatch = valid & (pred_labels != true_labels)            # (B, T)

        if not mismatch.any():
            return torch.tensor(0.0, device=device)

        h   = F.normalize(sent_vecs[mismatch], dim=-1)            # (M, D)
        tl  = true_labels[mismatch]                                # (M,)
        pl  = pred_labels[mismatch]                                # (M,)

        # Pull toward true prototype
        proto_true = F.normalize(proto_matrix[tl], dim=-1)        # (M, D)
        sim_true   = (h * proto_true).sum(dim=-1)                  # (M,)
        pull_loss  = (1.0 - sim_true).mean()

        # Push away from wrong prototype
        proto_pred = F.normalize(proto_matrix[pl], dim=-1)        # (M, D)
        sim_pred   = (h * proto_pred).sum(dim=-1)                  # (M,)
        push_loss  = F.relu(sim_pred - self.margin).mean()

        return self.weight * (pull_loss + push_loss)


# ═══════════════════════════════════════════════════════════
# PROTOTYPE EMISSION BIAS (batched)
# ═══════════════════════════════════════════════════════════
class PrototypeEmissionBias(nn.Module):
    def __init__(self):
        super().__init__()
        self.scale = nn.Parameter(torch.tensor(KG_PROTO_WEIGHT_INIT))

    def forward(self, sent_vecs, emissions, proto_matrix):
        """
        sent_vecs   : (B, T, D)
        emissions   : (B, T, C)
        proto_matrix: (C, D)  — some rows may be zero if label missing
        Returns biased emissions (B, T, C)
        """
        sv_norm = F.normalize(sent_vecs, dim=-1)                   # (B, T, D)
        pm_norm = F.normalize(proto_matrix, dim=-1)                # (C, D)
        # (B, T, D) × (D, C) → (B, T, C)
        sim = torch.matmul(sv_norm, pm_norm.T)
        # non-zero proto mask
        mask = (proto_matrix.abs().sum(dim=-1) > 0).float()        # (C,)
        return emissions + self.scale.abs() * sim * mask.unsqueeze(0).unsqueeze(0)


# ═══════════════════════════════════════════════════════════
# KG-AUGMENTED MODEL (fixed batched version)
# ═══════════════════════════════════════════════════════════
class KGAugmentedModelFast(nn.Module):
    def __init__(self, base_model, kg_index: BatchedKGIndex, rare_ids, class_weights=None):
        super().__init__()
        self.base        = base_model
        self.kg_index    = kg_index
        self.rare_set    = set(rare_ids)
        self.uncertainty = UncertaintyEstimator()
        self.contrastive = KGContrastiveLoss()
        self.proto_bias  = PrototypeEmissionBias()

        sent_dim = base_model.sent_out_dim  # 256
        ctx_dim  = base_model.ctx_out_dim   # 128

        self.gat_fusion = BatchedGraphAttentionFusion(emb_dim=sent_dim)
        self.fusion_classifier = nn.Sequential(
            nn.Dropout(DROPOUT), nn.Linear(ctx_dim, ctx_dim // 2),
            nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(ctx_dim // 2, NUM_LABELS),
        )
        self.fusion_crf = CRF(num_tags=NUM_LABELS, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            weight=class_weights, label_smoothing=LABEL_SMOOTHING, ignore_index=-100)

        # Precompute prototype matrix (stays fixed during training)
        self.register_buffer("proto_matrix", kg_index.get_prototype_matrix())

        # Rare-class prior correction (log-uniform offset to boost rare labels)
        rare_bias = torch.zeros(NUM_LABELS)
        for rid in rare_ids:
            rare_bias[rid] = 1.5   # additive log-space boost
        self.register_buffer("rare_bias", rare_bias)

    def _build_fusion_mask(self, emissions, lengths):
        """Build (B, T) bool mask: True where we want KG fusion."""
        B, T, _ = emissions.shape
        uncertain = self.uncertainty.is_uncertain(emissions)         # (B, T)
        top_lbl   = self.uncertainty.top_label(emissions)           # (B, T)

        # Always fuse for rare-class predictions
        is_rare = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
        for rid in self.rare_set:
            is_rare |= (top_lbl == rid)

        fusion_mask = uncertain | (RARE_ALWAYS_KG and is_rare)

        # Restrict to valid positions only
        if lengths is not None:
            for i, l in enumerate(lengths):
                fusion_mask[i, l:] = False

        return fusion_mask

    def forward(self, input_ids, attention_mask, token_type_ids, labels=None, lengths=None):
        device = input_ids.device

        # Step 1: base model
        sent_vecs, ctx_out, base_emissions = self.base.get_emissions(
            input_ids, attention_mask, token_type_ids, lengths=lengths)

        # Step 2: add rare-class bias + prototype bias to base emissions
        base_emissions = base_emissions + self.rare_bias.to(device)
        base_emissions = self.proto_bias(sent_vecs, base_emissions, self.proto_matrix)

        # Step 3: build fusion mask (pure tensor ops, no Python loops)
        fusion_mask = self._build_fusion_mask(base_emissions, lengths)  # (B, T)

        # Step 4: batched KG retrieval — ONE matmul for all sentences
        top_embs, top_sims, top_labels = self.kg_index.retrieve_batch(sent_vecs, KG_TOP_K)
        # top_embs: (B, T, K, D), top_sims: (B, T, K)

        # Step 5: batched graph attention fusion
        fused_sent = self.gat_fusion(sent_vecs, top_embs, top_sims, fusion_mask)
        # (B, T, D) — h_i + v_i where fusion_mask=True, else h_i

        # Step 6: context BiLSTM on fused vecs
        fused_drop = self.base.dropout(fused_sent)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                fused_drop, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.base.ctx_bilstm(packed)
            fused_ctx, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            fused_ctx, _ = self.base.ctx_bilstm(fused_drop)
        fused_ctx = self.base.dropout(fused_ctx)

        # Step 7: fused emissions + proto bias + rare bias
        fused_emissions = self.fusion_classifier(fused_ctx)
        fused_emissions = fused_emissions + self.rare_bias.to(device)
        fused_emissions = self.proto_bias(fused_sent, fused_emissions, self.proto_matrix)
        fused_emissions = torch.nan_to_num(fused_emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        # Mask
        if lengths is not None:
            B, T, _ = fused_emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=device)
            for i, l in enumerate(lengths): mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(fused_emissions.shape[:2], dtype=torch.bool, device=device)

        # Ensemble: 0.35 base + 0.65 fused (fused weighted higher for minority)
        combined = 0.35 * base_emissions + 0.65 * fused_emissions

        if labels is not None:
            safe = labels.clone(); safe[safe == -100] = 0
            base_crf_loss  = -self.base.crf(base_emissions, safe, mask=mask, reduction="mean")
            fused_crf_loss = -self.fusion_crf(fused_emissions, safe, mask=mask, reduction="mean")
            B2, T2, C = combined.shape
            ce_loss = self.ce_loss(combined.reshape(B2 * T2, C), labels.reshape(B2 * T2))

            # Contrastive (batched)
            with torch.no_grad():
                pred_labels = combined.argmax(dim=-1)
            contrastive_loss = self.contrastive(
                sent_vecs, pred_labels, labels, self.proto_matrix, device)

            loss = ((base_crf_loss + fused_crf_loss) / 2.0
                    + AUX_CE_WEIGHT * ce_loss + contrastive_loss)
            return loss, combined
        else:
            decoded = self.fusion_crf.decode(fused_emissions, mask=mask)
            return decoded, fused_emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_prec  = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    macro_rec   = recall_score   (all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec  = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    micro_rec   = recall_score   (all_trues, all_preds, average="micro",    zero_division=0)
    wt_prec     = precision_score(all_trues, all_preds, average="weighted", zero_division=0)
    wt_rec      = recall_score   (all_trues, all_preds, average="weighted", zero_division=0)
    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    per_class_metrics = {id2label[i]: {"f1": float(per_class_f1[i]),
                                        "precision": float(per_class_prec[i]),
                                        "recall": float(per_class_rec[i])}
                         for i in range(NUM_LABELS)}

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]
    cls_report = classification_report(str_trues, str_preds, labels=LABELS, digits=4, zero_division=0)
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec, "weighted_precision": wt_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec, "weighted_recall": wt_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


def count_parameters(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    fr = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"\n  Trainable: {tr:,} | Frozen: {fr:,}")
    return tr, fr


# ═══════════════════════════════════════════════════════════
# BASE TRAINER (Phase A)
# ═══════════════════════════════════════════════════════════
class BaseTrainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        pg = [{"params": list(self.model.bert.pooler.parameters()),
               "lr": BERT_LR, "weight_decay": WEIGHT_DECAY}]
        enc = self.model.bert.encoder.layer
        n   = len(enc)
        for i in range(n - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in enc[i].parameters() if p.requires_grad]
            if params: pg.append({"params": params, "lr": lr_i, "weight_decay": WEIGHT_DECAY})
        head_params = []
        for m in [self.model.sent_bilstm, self.model.mha_pooling,
                  self.model.sent_layer_norm, self.model.ctx_bilstm,
                  self.model.classifier, self.model.crf]:
            head_params.extend(list(m.parameters()))
        pg.append({"params": head_params, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY})
        return torch.optim.AdamW(pg)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        total, n = 0.0, 0
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); labels = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if not torch.isnan(loss): total += loss.item(); n += 1
        return total / max(1, n)

    def evaluate(self, dataset, rare_ids, split_name="dev", measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        t0 = time.time() if measure_inference_time else None
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype, labels=None, lengths=lengths)
                for i, seq in enumerate(decoded):
                    all_preds.extend(seq)
                    all_trues.extend(labels[i, :int(lengths[i].item())].tolist())
        if measure_inference_time and t0:
            ti = {"total_inference_time_s": time.time() - t0,
                  "latency_per_document_ms": (time.time()-t0)/max(1,len(dataset))*1000}
            with open(os.path.join(OUT_DIR, f"inference_{split_name}.json"), "w") as f:
                json.dump(ti, f, indent=2)
        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    def train(self, train_dataset, dev_dataset, rare_ids, tokenizer, num_epochs=NUM_EPOCHS_BASE):
        loader   = DataLoader(train_dataset, batch_size=BATCH_DOCS, shuffle=True, collate_fn=collate_rrc)
        opt      = self.build_optimizer()
        total_s  = len(loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        sched    = get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO * total_s), total_s)
        es       = EarlyStopping()
        history  = []
        best_f1, best_state = -1.0, None
        t0 = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            run_loss, n_steps = 0.0, 0
            ep_t = time.time()
            opt.zero_grad()
            for step, (ids, attn, ttype, labels, lengths) in enumerate(loader):
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); labels = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss): opt.zero_grad(); continue
                (loss / GRADIENT_ACCUMULATION_STEPS).backward()
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    opt.step(); sched.step(); opt.zero_grad()
                run_loss += loss.item(); n_steps += 1
            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                opt.step(); sched.step(); opt.zero_grad()

            ep_time = time.time() - ep_t
            avg_loss = run_loss / max(1, n_steps)
            val_loss = self.compute_val_loss(dev_dataset)
            val_m    = self.evaluate(dev_dataset, rare_ids)
            print(f"[Base] Epoch {epoch:03d}/{num_epochs} | "
                  f"train_loss: {avg_loss:.4f} | val_loss: {val_loss:.4f} | "
                  f"val_macro_f1: {val_m['macro_f1']:.4f} | "
                  f"val_rare_f1: {val_m['rare_f1']:.4f} | "
                  f"time: {ep_time:.1f}s | ES: {es.counter}/{es.patience}")
            history.append({"epoch": epoch, "phase": "base",
                             "train_loss": avg_loss, "val_loss": val_loss,
                             "val_macro_f1": val_m["macro_f1"],
                             "val_micro_f1": val_m["micro_f1"],
                             "val_weighted_f1": val_m["weighted_f1"],
                             "val_rare_f1": val_m["rare_f1"],
                             "val_accuracy": val_m["accuracy"],
                             "epoch_train_time_s": ep_time})
            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1 = val_m["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")
            if es.step(val_m["macro_f1"]): print(f"\n⏹ Early stopping at epoch {epoch}.\n"); break

        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "base_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            print(f"\n✔ Best base model restored (val_macro_f1={best_f1:.4f})")
        if tokenizer:
            self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
            tokenizer.save_pretrained(BEST_MODEL_DIR)
            torch.save(best_state or self.model.state_dict(),
                       os.path.join(BEST_MODEL_DIR, "base_model.bin"))
        return pd.DataFrame(history), time.time() - t0


# ═══════════════════════════════════════════════════════════
# KG BUILDER
# ═══════════════════════════════════════════════════════════
@torch.no_grad()
def build_kg_store(base_model, train_docs, tokenizer, device=DEVICE) -> KGStore:
    print("\n🔨 Building KG node store from training embeddings ...")
    base_model.eval().to(device)
    kg = KGStore(emb_dim=base_model.sent_out_dim)
    dataset = RRCDataset(train_docs, tokenizer)
    loader  = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=collate_rrc)
    for idx, (ids, attn, ttype, labels, lengths) in enumerate(loader):
        ids = ids.to(device); attn = attn.to(device); ttype = ttype.to(device)
        sv  = base_model.encode_sentences(ids, attn, ttype).squeeze(0)  # (T, D)
        n   = int(lengths[0].item())
        kg.add_nodes(sv[:n].cpu(), labels[0, :n].tolist())
        if (idx + 1) % 100 == 0:
            print(f"  {idx+1}/{len(loader)} docs processed")
    kg.save(os.path.join(OUT_DIR, "kg_store.json"))
    total = sum(len(v) for v in kg.nodes.values())
    print(f"  KGStore built: {total} nodes across {len(kg.nodes)} labels")
    return kg


# ═══════════════════════════════════════════════════════════
# KG TRAINER (Phase B) — 60 epochs, NO early stopping
# ═══════════════════════════════════════════════════════════
class KGTrainerFast:
    def __init__(self, kg_model: KGAugmentedModelFast, device=DEVICE):
        self.model  = kg_model.to(device)
        self.device = device

    def build_optimizer(self):
        new_params = (list(self.model.gat_fusion.parameters()) +
                      list(self.model.fusion_classifier.parameters()) +
                      list(self.model.fusion_crf.parameters()) +
                      list(self.model.proto_bias.parameters()))
        base_trainable = [p for p in self.model.base.parameters() if p.requires_grad]
        return torch.optim.AdamW([
            {"params": new_params,     "lr": HEAD_LR,  "weight_decay": WEIGHT_DECAY},
            {"params": base_trainable, "lr": BERT_LR,  "weight_decay": WEIGHT_DECAY},
        ])

    def evaluate(self, dataset, rare_ids, split_name="dev", measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        t0 = time.time() if measure_inference_time else None
        with torch.no_grad():
            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); lengths = lengths.to(self.device)
                decoded, _ = self.model(ids, attn, ttype, labels=None, lengths=lengths)
                for i, seq in enumerate(decoded):
                    all_preds.extend(seq)
                    all_trues.extend(labels[i, :int(lengths[i].item())].tolist())
        if measure_inference_time and t0:
            ti = {"total_inference_time_s": time.time() - t0,
                  "latency_per_document_ms": (time.time()-t0)/max(1,len(dataset))*1000,
                  "throughput_sentences_per_s": len(all_trues)/max(1e-9, time.time()-t0)}
            with open(os.path.join(OUT_DIR, f"kg_inference_{split_name}.json"), "w") as f:
                json.dump(ti, f, indent=2)
        return compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

    def train(self, train_dataset, dev_dataset, rare_ids, num_epochs=NUM_EPOCHS_KG):
        loader   = DataLoader(train_dataset, batch_size=BATCH_DOCS, shuffle=True, collate_fn=collate_rrc)
        opt      = self.build_optimizer()
        total_s  = len(loader) * num_epochs
        sched    = get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO * total_s), total_s)
        history  = []
        best_f1, best_state = -1.0, None
        t0 = time.time()

        print(f"\n  Phase B: {num_epochs} epochs, NO early stopping.\n")

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            run_loss, n_steps = 0.0, 0
            ep_t = time.time()
            opt.zero_grad()

            for ids, attn, ttype, labels, lengths in loader:
                ids = ids.to(self.device); attn = attn.to(self.device)
                ttype = ttype.to(self.device); labels = labels.to(self.device)
                lengths = lengths.to(self.device)
                loss, _ = self.model(ids, attn, ttype, labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss): opt.zero_grad(); continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                opt.step(); sched.step(); opt.zero_grad()
                run_loss += loss.item(); n_steps += 1

            ep_time  = time.time() - ep_t
            avg_loss = run_loss / max(1, n_steps)
            val_m    = self.evaluate(dev_dataset, rare_ids)

            print(f"[KG]  Epoch {epoch:02d}/{num_epochs} | "
                  f"train_loss: {avg_loss:.4f} | "
                  f"val_macro_f1: {val_m['macro_f1']:.4f} | "
                  f"val_rare_f1: {val_m['rare_f1']:.4f} | "
                  f"time: {ep_time:.1f}s")

            history.append({"epoch": epoch, "phase": "kg",
                             "train_loss": avg_loss,
                             "val_macro_f1": val_m["macro_f1"],
                             "val_rare_f1": val_m["rare_f1"],
                             "val_accuracy": val_m["accuracy"],
                             "epoch_train_time_s": ep_time})

            if val_m["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1 = val_m["macro_f1"]
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best KG val_macro_f1={best_f1:.4f}")

        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "kg_history.csv"), index=False)
        if best_state:
            self.model.load_state_dict(best_state)
            torch.save(best_state, os.path.join(BEST_MODEL_DIR, "kg_model.bin"))
            print(f"\n✔ Best KG model saved (val_macro_f1={best_f1:.4f})")
        return pd.DataFrame(history), time.time() - t0


# ═══════════════════════════════════════════════════════════
# VISUALISATION
# ═══════════════════════════════════════════════════════════
def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            if tick.get_text() in rare_labels: tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix (KG-Fast)")
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue" for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12); ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1 (KG-Fast)")
    ax.grid(True, alpha=0.3, axis="x"); plt.tight_layout()
    path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def plot_combined_history(base_df, kg_df):
    all_df = pd.concat([base_df, kg_df], ignore_index=True)
    all_df["global_epoch"] = range(1, len(all_df) + 1)
    boundary = len(base_df)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(all_df["global_epoch"], all_df["train_loss"], marker="o", markersize=3)
    axes[0].axvline(boundary, color="red", linestyle="--", label="Phase B start")
    axes[0].set_title("Combined Training Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(all_df["global_epoch"], all_df["val_macro_f1"], label="Macro-F1", marker="o", markersize=3)
    axes[1].plot(all_df["global_epoch"], all_df["val_rare_f1"], label="Rare-F1", marker="s", markersize=3)
    axes[1].axvline(boundary, color="red", linestyle="--", label="Phase B start")
    axes[1].set_title("Val F1 (Base → KG-Fast)"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(OUT_DIR, "combined_training_curves.png")
    plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


def print_metrics_table(dev_m, test_m, base_time=None, kg_time=None, total_trainable=None):
    rows = [("Accuracy","accuracy"),("Macro-F1","macro_f1"),("Micro-F1","micro_f1"),
            ("Weighted-F1","weighted_f1"),("Rare/Minority F1","rare_f1"),
            ("Macro-Precision","macro_precision"),("Macro-Recall","macro_recall")]
    print("\n" + "=" * 72)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + CRF + KG-Fast)")
    print("=" * 72)
    if total_trainable: print(f"  Trainable Parameters : {total_trainable:,}")
    if base_time:       print(f"  Phase A time         : {base_time/60:.1f} min")
    if kg_time:         print(f"  Phase B time         : {kg_time/60:.1f} min")
    print("-" * 72)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 72)
    for label, key in rows:
        print(f"  {label:<28} {dev_m[key]:>12.4f} {test_m[key]:>12.4f}")
    print("=" * 72)
    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 64)
    print(f"  {'Label':<22} {'F1-Dev':>9} {'F1-Test':>9} {'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 64)
    for lbl in LABELS:
        dv = dev_m["per_class_metrics"][lbl]; ts = test_m["per_class_metrics"][lbl]
        print(f"  {lbl:<22} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 64)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture: InLegalBERT + BiLSTM + MHA + CRF + KG-Fast (batched GPU retrieval)")
    print("\nKey fixes vs slow version:")
    print("  ✓ Batched GPU KG retrieval (1 matmul per forward, no per-sentence loops)")
    print("  ✓ KG index capped at 500 nodes/label (memory efficient)")
    print("  ✓ Phase B: 60 epochs, NO early stopping")
    print("  ✓ PRE_NOT_RELIED weight x10, ARG_RESPONDENT x6")
    print("  ✓ Rare-class emission bias (+1.5 additive log boost)")
    print("  ✓ Batched contrastive loss + prototype anchoring\n")

    print("Loading JSONL files ...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    class_weights = compute_class_weights(train_docs, rare_ids).to(DEVICE)

    pd.DataFrame([{"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
                  for l in LABELS]).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("\nLoading tokenizer ...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ════ PHASE A ════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A: Base Model Training")
    print("=" * 60)

    base_model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name=INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden=SENT_LSTM_HIDDEN, sent_lstm_layers=SENT_LSTM_LAYERS,
        ctx_lstm_hidden=CTX_LSTM_HIDDEN,   ctx_lstm_layers=CTX_LSTM_LAYERS,
        mha_heads=MHA_HEADS, mha_dropout=MHA_DROPOUT,
        num_labels=NUM_LABELS, dropout=DROPOUT, freeze_layers=BERT_FREEZE_LAYERS,
        class_weights=class_weights,
    )
    base_trainer = BaseTrainer(base_model, device=DEVICE)
    base_hist_df, base_time = base_trainer.train(
        train_dataset, dev_dataset, rare_ids, tokenizer=tokenizer,
        num_epochs=NUM_EPOCHS_BASE,
    )

    # ════ BUILD KG ═══════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE A→B: Building KG Node Store")
    print("=" * 60)
    kg_path = os.path.join(OUT_DIR, "kg_store.json")
    if os.path.exists(kg_path):
        print("  Found existing KG, loading ...")
        kg_store = KGStore.load(kg_path, emb_dim=base_model.sent_out_dim)
    else:
        kg_store = build_kg_store(base_model, train_docs, tokenizer, DEVICE)

    # Build batched GPU index (the critical step — fast retrieval)
    print("\n  Building batched GPU index ...")
    kg_index = BatchedKGIndex(kg_store.nodes, emb_dim=base_model.sent_out_dim, device=DEVICE)

    # ════ PHASE B ════════════════════════════════════════
    print("\n" + "=" * 60)
    print("PHASE B: KG-Augmented Fine-Tuning (60 epochs, no early stopping)")
    print("=" * 60)

    kg_model = KGAugmentedModelFast(
        base_model=base_model, kg_index=kg_index,
        rare_ids=rare_ids, class_weights=class_weights,
    )
    total_trainable, _ = count_parameters(kg_model)
    kg_trainer = KGTrainerFast(kg_model, device=DEVICE)
    kg_hist_df, kg_time = kg_trainer.train(
        train_dataset, dev_dataset, rare_ids, num_epochs=NUM_EPOCHS_KG,
    )
    plot_combined_history(base_hist_df, kg_hist_df)

    # ════ EVALUATION ════════════════════════════════════
    print("\nEvaluating on Dev set ...")
    dev_m = kg_trainer.evaluate(dev_dataset, rare_ids, split_name="dev",
                                 measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_m['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_m['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_m['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + KG-Fast\n")
        f.write(f"Rare classes: {rare_labels}\n\n")
        f.write(dev_m["cls_report"])
    save_confusion_matrix(dev_m["cm"], "dev", rare_labels)
    save_per_class_f1_chart(dev_m["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set ...")
    test_m = kg_trainer.evaluate(test_dataset, rare_ids, split_name="test",
                                  measure_inference_time=True)
    print(f"  Test Accuracy : {test_m['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_m['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_m['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + KG-Fast\n")
        f.write(f"Rare classes: {rare_labels}\n\n")
        f.write(test_m["cls_report"])
    save_confusion_matrix(test_m["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_m["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({"true": [id2label[x] for x in test_m["all_trues"]],
                  "pred": [id2label[x] for x in test_m["all_preds"]]
                  }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    scalar_keys = ["macro_f1","micro_f1","weighted_f1","rare_f1",
                   "macro_precision","micro_precision","weighted_precision","rare_precision",
                   "macro_recall","micro_recall","weighted_recall","rare_recall","accuracy"]
    summary = {
        "model": "InLegalBERT + BiLSTM + MHA + CRF + KG-Fast",
        "fixes": ["batched_gpu_retrieval","no_early_stopping_phase_b",
                  "rare_emission_bias","boosted_minority_weights",
                  "batched_contrastive_loss","prototype_anchoring"],
        "timing": {"phase_a_s": base_time, "phase_b_s": kg_time,
                   "total_s": base_time + kg_time},
        "rare_classes": rare_labels,
        "dev":  {k: dev_m[k]  for k in scalar_keys},
        "test": {k: test_m[k] for k in scalar_keys},
        "per_class_dev":  dev_m["per_class_metrics"],
        "per_class_test": test_m["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print_metrics_table(dev_m, test_m, base_time=base_time, kg_time=kg_time,
                        total_trainable=total_trainable)
    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture: InLegalBERT + BiLSTM + MHA + CRF + KG-Fast (batched GPU retrieval)

Key fixes vs slow version:
  ✓ Batched GPU KG retrieval (1 matmul per forward, no per-sentence loops)
  ✓ KG index capped at 500 nodes/label (memory efficient)
  ✓ Phase B: 60 epochs, NO early stopping
  ✓ PRE_NOT_RELIED weight x10, ARG_RESPONDENT x6
  ✓ Rare-class emission bias (+1.5 additive log boost)
  ✓ Batched contrastive loss + prototype anchoring

Loading JSONL files ...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED        

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT frozen: embeddings + layers 0-7.
🔥 BERT trainable: layers 8-11 + pooler.

[Base] Epoch 001/60 | train_loss: 287.9091 | val_loss: 212.1917 | val_macro_f1: 0.0391 | val_rare_f1: 0.0000 | time: 35.7s | ES: 0/10
  ✔ New best val_macro_f1=0.0391
[Base] Epoch 002/60 | train_loss: 235.2646 | val_loss: 171.9216 | val_macro_f1: 0.0813 | val_rare_f1: 0.0000 | time: 36.7s | ES: 0/10
  ✔ New best val_macro_f1=0.0813
[Base] Epoch 003/60 | train_loss: 199.8910 | val_loss: 127.9615 | val_macro_f1: 0.2023 | val_rare_f1: 0.0551 | time: 37.0s | ES: 0/10
  ✔ New best val_macro_f1=0.2023
[Base] Epoch 004/60 | train_loss: 159.2420 | val_loss: 98.0553 | val_macro_f1: 0.2836 | val_rare_f1: 0.1218 | time: 36.1s | ES: 0/10
  ✔ New best val_macro_f1=0.2836
[Base] Epoch 005/60 | train_loss: 132.7343 | val_loss: 87.0139 | val_macro_f1: 0.2760 | val_rare_f1: 0.1088 | time: 36.5s | ES: 0/10
[Base] Epoch 006/60 | train_loss: 123.9431 | val_loss: 84.1318 | val_macro_f1: 0.3031 | val_rare_f1: 0.1493 | time: 